In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [2]:
from DLRM.models import (
    DCNV2,
    FieldAwareTransformerRanker,
    HiformerRanker,
    RankMixerRanker,
    TokenMixerLargeRanker,
)
from DLRM.training import train_model
from DLRM.models.common import PiecewiseLinearEncoder


def build_dcnv2(embedding_size, train_listens_dense_million, deep_network="mlp"):
    dense_encoder = PiecewiseLinearEncoder.from_dataset(
        dense_train_df=train_listens_dense_million,
        n_bins=16,
        train_df_slice=1_000_000,
    )

    dense_input_size = sum(dense_encoder.n_bins)
    num_hashes = 2

    input_size = dense_input_size + 4 * num_hashes * embedding_size
    return DCNV2(
        embedding_size=embedding_size,
        cross_layers=3,
        deep_units=[256, 128],
        input_size=input_size,
        dense_train_df=train_listens_dense_million,
        n_bins=16,
        train_df_slice=1_000_000,
        cardinality=65536,
        num_experts=4,
        low_rank=32,
        deep_network=deep_network,
    )


def build_hiformer(embedding_size, train_listens_dense_million):
    return HiformerRanker(
        embedding_size=embedding_size,
        dense_train_df=train_listens_dense_million,
        n_bins=16,
        train_df_slice=1_000_000,
        cardinality=65536,
        output_size=2,
        n_dense_tokens=4,
        model_dim=64,
        num_heads=4,
        num_layers=1,
        rank_qk=32,
        rank_v=64,
        dropout=0.1,
    )


def build_fat(embedding_size, train_listens_dense_million):
    return FieldAwareTransformerRanker(
        embedding_size=embedding_size,
        dense_train_df=train_listens_dense_million,
        n_bins=16,
        train_df_slice=1_000_000,
        cardinality=65536,
        output_size=2,
        n_dense_tokens=4,
        model_dim=64,
        num_layers=2,
        num_heads=4,
        head_dim=16,
        num_bases=16,
        meta_dim=32,
        top_k=3,
        dropout=0.1,
    )


def build_rankmixer(embedding_size, train_listens_dense_million):
    return RankMixerRanker(
        embedding_size=embedding_size,
        dense_train_df=train_listens_dense_million,
        n_bins=16,
        train_df_slice=1_000_000,
        cardinality=65536,
        output_size=2,
        n_dense_tokens=4,
        model_dim=64,
        num_layers=2,
        ffn_mult=4,
        dropout=0.1,
    )


def build_tokenmixer(embedding_size, train_listens_dense_million):
    return TokenMixerLargeRanker(
        embedding_size=embedding_size,
        dense_train_df=train_listens_dense_million,
        n_bins=16,
        train_df_slice=1_000_000,
        cardinality=65536,
        output_size=2,
        n_dense_tokens=4,
        model_dim=64,
        num_layers=4,
        num_mixed_tokens=8,
        expansion=4,
        dropout=0.1,
    )


In [ ]:
import gc
import warnings

import polars as pl
from huggingface_hub import hf_hub_download

warnings.filterwarnings("ignore")

from DLRM.data import (
    DatasetConfig,
    YambdaDataset,
    join_item_artist_album,
    temporal_train_test_split,
    RankerDataset,
    MultihashTransform,
)

from DLRM.config import (
    DENSE_COLUMNS,
    MULTIVALENT_COLUMNS,
    SPARSE_COLUMNS,
    LABEL_COLUMNS,
)

path = hf_hub_download(
    repo_id="matfu21/yambda-50m-lag-features",
    repo_type="dataset",
    filename="listens.parquet",
)

listens = pl.read_parquet(path)
dataset_config = DatasetConfig()

yambda_dataset = YambdaDataset(
    dataset_type=dataset_config.dataset_type,
    dataset_size=dataset_config.dataset_size,
)

albums = yambda_dataset.album_item_mapping().to_polars()
artists = yambda_dataset.artist_item_mapping().to_polars()

listens = join_item_artist_album(
    listens=listens,
    artists=artists,
    albums=albums,
)

del albums, artists
gc.collect()

train_listens, test_listens = temporal_train_test_split(
    listens,
    test_quantile=0.9,
)

del listens
gc.collect()

from torch.utils.data import DataLoader

BATCH_SIZE = 4096
CARDINALITY = 65536

multihash_transform = MultihashTransform(
    sparse_features_config={
        "item_id": [11, 22],
        "uid": [33, 44],
    },
    sparse_features_name="sparse_features",
    multivalent_features_config={
        "artist_ids": [55, 66],
        "album_ids": [77, 88],
    },
    multivalent_features_name="multivalent_features",
    cardinality=CARDINALITY,
)

train_listens_dense_million = train_listens.select(DENSE_COLUMNS).slice(0, 1_000_000)

train_dataset = RankerDataset(
    df=train_listens,
    transforms=[multihash_transform],
    label_columns=list(LABEL_COLUMNS),
    dense_columns=list(DENSE_COLUMNS),
    sparse_columns=list(SPARSE_COLUMNS),
    multivalent_columns=list(MULTIVALENT_COLUMNS),
    batch_size=BATCH_SIZE,
)

test_dataset = RankerDataset(
    df=test_listens,
    transforms=[multihash_transform],
    label_columns=list(LABEL_COLUMNS),
    dense_columns=list(DENSE_COLUMNS),
    sparse_columns=list(SPARSE_COLUMNS),
    multivalent_columns=list(MULTIVALENT_COLUMNS),
    batch_size=BATCH_SIZE,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    collate_fn=lambda batch: batch[0],
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=lambda batch: batch[0],
)
embedding_size = 16


In [7]:
model_dcnv2_mlp = build_dcnv2(embedding_size, train_listens_dense_million, deep_network="mlp")
metrics_dcnv2_mlp = train_model(model_dcnv2_mlp, train_loader, test_loader, epochs=3, lr=1e-3, train_log_every=100)

epoch=1 step=100 loss=0.322996 loss_like=0.098448 loss_full=0.547544
epoch=1 step=200 loss=0.301450 loss_like=0.082755 loss_full=0.520145
epoch=1 step=300 loss=0.288600 loss_like=0.073729 loss_full=0.503471
epoch=1 step=400 loss=0.286727 loss_like=0.072027 loss_full=0.501428
epoch=1 step=500 loss=0.283969 loss_like=0.070200 loss_full=0.497738
epoch=1 step=600 loss=0.282817 loss_like=0.070368 loss_full=0.495266
epoch=1 step=700 loss=0.282093 loss_like=0.070321 loss_full=0.493865
epoch=1 step=800 loss=0.282268 loss_like=0.071111 loss_full=0.493424
epoch=1 step=900 loss=0.283015 loss_like=0.071148 loss_full=0.494882
epoch=1 step=1000 loss=0.281220 loss_like=0.070824 loss_full=0.491617
epoch=1 step=1100 loss=0.280498 loss_like=0.070845 loss_full=0.490151
epoch=1 step=1200 loss=0.279000 loss_like=0.069507 loss_full=0.488493
epoch=1 step=1300 loss=0.278208 loss_like=0.069289 loss_full=0.487127
epoch=1 step=1400 loss=0.278370 loss_like=0.068644 loss_full=0.488096
epoch=1 step=1500 loss=0.2779

In [8]:
model_dcnv2_resnet = build_dcnv2(embedding_size, train_listens_dense_million, deep_network="resnet")
metrics_dcnv2_resnet = train_model(model_dcnv2_resnet, train_loader, test_loader, epochs=3, lr=1e-3, train_log_every=100)

epoch=1 step=100 loss=0.339211 loss_like=0.113222 loss_full=0.565200
epoch=1 step=200 loss=0.321789 loss_like=0.106902 loss_full=0.536676
epoch=1 step=300 loss=0.307633 loss_like=0.096130 loss_full=0.519137
epoch=1 step=400 loss=0.302604 loss_like=0.093227 loss_full=0.511980
epoch=1 step=500 loss=0.297809 loss_like=0.087333 loss_full=0.508286
epoch=1 step=600 loss=0.293963 loss_like=0.084553 loss_full=0.503373
epoch=1 step=700 loss=0.293036 loss_like=0.083377 loss_full=0.502695
epoch=1 step=800 loss=0.289714 loss_like=0.081484 loss_full=0.497945
epoch=1 step=900 loss=0.289206 loss_like=0.080447 loss_full=0.497965
epoch=1 step=1000 loss=0.287060 loss_like=0.078656 loss_full=0.495464
epoch=1 step=1100 loss=0.287299 loss_like=0.078152 loss_full=0.496447
epoch=1 step=1200 loss=0.286070 loss_like=0.076153 loss_full=0.495987
epoch=1 step=1300 loss=0.284833 loss_like=0.075412 loss_full=0.494255
epoch=1 step=1400 loss=0.283738 loss_like=0.074343 loss_full=0.493133
epoch=1 step=1500 loss=0.2834

In [9]:
model_dcnv2_densenet = build_dcnv2(embedding_size, train_listens_dense_million, deep_network="densenet")
metrics_dcnv2_densenet = train_model(model_dcnv2_densenet, train_loader, test_loader, epochs=3, lr=1e-3, train_log_every=100)

epoch=1 step=100 loss=0.342792 loss_like=0.123216 loss_full=0.562368
epoch=1 step=200 loss=0.310093 loss_like=0.093210 loss_full=0.526975
epoch=1 step=300 loss=0.296236 loss_like=0.086363 loss_full=0.506109
epoch=1 step=400 loss=0.288690 loss_like=0.079084 loss_full=0.498295
epoch=1 step=500 loss=0.287335 loss_like=0.076984 loss_full=0.497686
epoch=1 step=600 loss=0.286264 loss_like=0.075601 loss_full=0.496927
epoch=1 step=700 loss=0.285854 loss_like=0.074827 loss_full=0.496882
epoch=1 step=800 loss=0.285026 loss_like=0.075792 loss_full=0.494259
epoch=1 step=900 loss=0.282524 loss_like=0.074725 loss_full=0.490323
epoch=1 step=1000 loss=0.280241 loss_like=0.073595 loss_full=0.486886
epoch=1 step=1100 loss=0.280003 loss_like=0.072465 loss_full=0.487542
epoch=1 step=1200 loss=0.279163 loss_like=0.072366 loss_full=0.485961
epoch=1 step=1300 loss=0.279129 loss_like=0.073417 loss_full=0.484842
epoch=1 step=1400 loss=0.279187 loss_like=0.072980 loss_full=0.485393
epoch=1 step=1500 loss=0.2782

In [10]:
model_hiformer = build_hiformer(embedding_size, train_listens_dense_million)
metrics_hiformer = train_model(model_hiformer, train_loader, test_loader, epochs=3, lr=1e-3, train_log_every=100)

epoch=1 step=100 loss=0.308587 loss_like=0.068954 loss_full=0.548220
epoch=1 step=200 loss=0.298236 loss_like=0.077826 loss_full=0.518645
epoch=1 step=300 loss=0.288114 loss_like=0.076153 loss_full=0.500074
epoch=1 step=400 loss=0.289916 loss_like=0.076188 loss_full=0.503644
epoch=1 step=500 loss=0.292473 loss_like=0.077491 loss_full=0.507454
epoch=1 step=600 loss=0.291494 loss_like=0.078623 loss_full=0.504364
epoch=1 step=700 loss=0.287881 loss_like=0.077358 loss_full=0.498403
epoch=1 step=800 loss=0.287335 loss_like=0.076622 loss_full=0.498049
epoch=1 step=900 loss=0.286369 loss_like=0.075325 loss_full=0.497412
epoch=1 step=1000 loss=0.286373 loss_like=0.075955 loss_full=0.496790
epoch=1 step=1100 loss=0.286511 loss_like=0.076674 loss_full=0.496348
epoch=1 step=1200 loss=0.285866 loss_like=0.075950 loss_full=0.495783
epoch=1 step=1300 loss=0.284961 loss_like=0.075032 loss_full=0.494890
epoch=1 step=1400 loss=0.283142 loss_like=0.074235 loss_full=0.492048
epoch=1 step=1500 loss=0.2840

In [ ]:
model_fat = build_fat(embedding_size, train_listens_dense_million)
metrics_fat = train_model(model_fat, train_loader, test_loader, epochs=3, lr=1e-3, train_log_every=100)

epoch=1 step=100 loss=0.340584 loss_like=0.138685 loss_full=0.542483
epoch=1 step=200 loss=0.320682 loss_like=0.109834 loss_full=0.531530
epoch=1 step=300 loss=0.308948 loss_like=0.098840 loss_full=0.519055
epoch=1 step=400 loss=0.300967 loss_like=0.092228 loss_full=0.509707
epoch=1 step=500 loss=0.295018 loss_like=0.085727 loss_full=0.504310
epoch=1 step=600 loss=0.291820 loss_like=0.083819 loss_full=0.499822
epoch=1 step=700 loss=0.288336 loss_like=0.081441 loss_full=0.495231
epoch=1 step=800 loss=0.285248 loss_like=0.079386 loss_full=0.491109
epoch=1 step=900 loss=0.284948 loss_like=0.079019 loss_full=0.490876
epoch=1 step=1000 loss=0.284993 loss_like=0.078392 loss_full=0.491595
epoch=1 step=1100 loss=0.284063 loss_like=0.076535 loss_full=0.491592
epoch=1 step=1200 loss=0.281736 loss_like=0.075411 loss_full=0.488061
epoch=1 step=1300 loss=0.280839 loss_like=0.074378 loss_full=0.487300
epoch=1 step=1400 loss=0.280690 loss_like=0.074460 loss_full=0.486919
epoch=1 step=1500 loss=0.2802

In [12]:
model_rankmixer = build_rankmixer(embedding_size, train_listens_dense_million)
metrics_rankmixer = train_model(model_rankmixer, train_loader, test_loader, epochs=3, lr=1e-3, train_log_every=100)

epoch=1 step=100 loss=0.344334 loss_like=0.114652 loss_full=0.574017
epoch=1 step=200 loss=0.313991 loss_like=0.092810 loss_full=0.535173
epoch=1 step=300 loss=0.305365 loss_like=0.091103 loss_full=0.519627
epoch=1 step=400 loss=0.294501 loss_like=0.083618 loss_full=0.505385
epoch=1 step=500 loss=0.287450 loss_like=0.078368 loss_full=0.496533
epoch=1 step=600 loss=0.288841 loss_like=0.077541 loss_full=0.500141
epoch=1 step=700 loss=0.287989 loss_like=0.076248 loss_full=0.499730
epoch=1 step=800 loss=0.285079 loss_like=0.075666 loss_full=0.494491
epoch=1 step=900 loss=0.284031 loss_like=0.074937 loss_full=0.493125
epoch=1 step=1000 loss=0.282840 loss_like=0.074326 loss_full=0.491355
epoch=1 step=1100 loss=0.282747 loss_like=0.073745 loss_full=0.491749
epoch=1 step=1200 loss=0.281261 loss_like=0.072847 loss_full=0.489675
epoch=1 step=1300 loss=0.280700 loss_like=0.073382 loss_full=0.488018
epoch=1 step=1400 loss=0.279939 loss_like=0.072202 loss_full=0.487676
epoch=1 step=1500 loss=0.2806

In [13]:
model_tokenmixer = build_tokenmixer(embedding_size, train_listens_dense_million)
metrics_tokenmixer = train_model(model_tokenmixer, train_loader, test_loader, epochs=3, lr=1e-3, train_log_every=100)

epoch=1 step=100 loss=0.311521 loss_like=0.101862 loss_full=0.521179
epoch=1 step=200 loss=0.297757 loss_like=0.090989 loss_full=0.504526
epoch=1 step=300 loss=0.290531 loss_like=0.081192 loss_full=0.499870
epoch=1 step=400 loss=0.289782 loss_like=0.079670 loss_full=0.499893
epoch=1 step=500 loss=0.289003 loss_like=0.077141 loss_full=0.500864
epoch=1 step=600 loss=0.286383 loss_like=0.077985 loss_full=0.494780
epoch=1 step=700 loss=0.281861 loss_like=0.076054 loss_full=0.487667
epoch=1 step=800 loss=0.280953 loss_like=0.074221 loss_full=0.487685
epoch=1 step=900 loss=0.280815 loss_like=0.073250 loss_full=0.488381
epoch=1 step=1000 loss=0.279432 loss_like=0.072224 loss_full=0.486640
epoch=1 step=1100 loss=0.279362 loss_like=0.071602 loss_full=0.487122
epoch=1 step=1200 loss=0.278244 loss_like=0.071604 loss_full=0.484883
epoch=1 step=1300 loss=0.277845 loss_like=0.071950 loss_full=0.483740
epoch=1 step=1400 loss=0.278125 loss_like=0.071324 loss_full=0.484925
epoch=1 step=1500 loss=0.2781

In [7]:
metrics = {
    "dcnv2 + mlp": metrics_dcnv2_mlp,
    "dcnv2 + resnet": metrics_dcnv2_resnet,
    "dcnv2 + densenet": metrics_dcnv2_densenet,
    "hiformer": metrics_hiformer,
    "field-aware transformer": metrics_fat,
    "rankmixer": metrics_rankmixer,
    "tokenmixer Large": metrics_tokenmixer,
}

metrics_df = pl.DataFrame(
    {
        "model": list(metrics.keys()),
        "e_task": [m["ne_e_task"] for m in metrics.values()],
        "c_task": [m["ne_c_task"] for m in metrics.values()],
    }
)

In [8]:
metrics_df

model,e_task,c_task
str,f64,f64
"""dcnv2 + mlp""",0.790818,0.73509
"""dcnv2 + resnet""",0.798482,0.735322
"""dcnv2 + densenet""",0.800486,0.737465
"""hiformer""",0.790128,0.735459
"""field-aware transformer""",0.804359,0.736128
"""rankmixer""",0.793621,0.732765
"""tokenmixer Large""",0.790171,0.733556
